In [5]:
%matplotlib inline
import trimesh
from matplotlib import pyplot as plt
from pcg_gazebo.simulation.properties import Mesh
from pcg_gazebo.simulation import create_object, SimulationModel
from pcg_gazebo.task_manager import get_rostopic_list


from pcg_gazebo.simulation import create_object

# If there is a Gazebo instance running, you can spawn the box into the simulation
from pcg_gazebo.task_manager import Server
# First create a simulation server
server = Server()
# Create a simulation manager named default
server.create_simulation('default', ros_port=11311, gazebo_port=11345)
simulation = server.get_simulation('default')
# Run an instance of the empty.world scenario
# This is equivalent to run
#      roslaunch gazebo_ros empty_world.launch
# with all default parameters
simulation.create_gazebo_empty_world_task()
simulation.create_rqt_task()
# A task named 'gazebo' the added to the tasks list
print(simulation.get_task_list())
# But it is still not running
print('Is Gazebo running: {}'.format(simulation.is_task_running('gazebo')))
# Run Gazebo
simulation.run_all_tasks()

from pcg_gazebo.generators import WorldGenerator
import random
# Create a Gazebo proxy
gazebo_proxy = simulation.get_gazebo_proxy()

# Use the generator to spawn the model to the Gazebo instance running at the moment
generator = WorldGenerator(gazebo_proxy=gazebo_proxy)

['rqt', 'gazebo']
Is Gazebo running: False


In [ ]:
# Office chairs
import math
model = SimulationModel.from_gazebo_model('office_chair')

generator.spawn_model(
    model=model, 
    robot_namespace='office_chair',
    pos=[random.random() * 10, random.random() * 2, 0],
    rot=[0, 0, (random.random() - 0.5) * math.pi],
    replace=True)



In [ ]:
# Add camera sensor
camera_sensor = SimulationModel(name='camera_standalone')
camera_sensor.static = True

camera_sensor.add_camera_sensor(
    add_visual=True, 
    add_collision=True, 
    add_ros_plugin=True,
    visualize=True,
    mass=0.01,
    size=[0.1, 0.1, 0.1],
    link_shape='cuboid',
    link_name='camera_link',
    image_width=1280,
    image_height=960)

print(camera_sensor.to_sdf())

# Spawn camera standalone model
generator.spawn_model(
    model=camera_sensor, 
    robot_namespace='camera_standalone',
    pos=[0, 0, 0.3])

print('List of ROS topics:')
for topic in simulation.get_rostopic_list():
    print(' - ' + topic)

In [ ]:
# End the simulation by killing the Gazebo task
simulation.kill_all_tasks()